# Test notebook for DPD funcionality

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
from flowermd.library import PPS

molecules = PPS(num_mols=50, lengths=50)
molecules.coarse_grain(beads={"A": "c1cc(S)ccc1"})
molecules.bond_length

In [ ]:
from flowermd.library import RandomWalk
import unyt as u

system = RandomWalk(
    molecules=molecules,
    density=1.32*u.Unit("g/cm**3"),
    bond_length=1.4226,
    buffer=0.58
)


In [ ]:
system.box.Lx

In [ ]:
from flowermd.library import DPD
A = 2000
gamma = 1500
kT = 1.0
r_cut = 1.4226
bond_k=50000
bond_r0=1.4226
dpd_ff = DPD(
        A=A,
        gamma=gamma,
        kT=kT,
        r_cut=r_cut,
        bond_k=bond_k,
        bond_r0=bond_r0)

In [ ]:
system.to_gsd('random_walk_test.gsd')

In [ ]:
hoomd_forces = system.hoomd_forcefield
hoomd_forces

In [ ]:
from flowermd.library.simulations.dpd_init import DPDInit

sim = DPDInit(
    initial_state=system.hoomd_snapshot,
    forcefield=dpd_ff.hoomd_forces,
    gsd_write_freq=100,
    log_write_freq=100,
    A=A,
    r_cut=r_cut,
    r=1.2,
    N=molecules.n_particles,
    box=system.box.Lx,
    sim_steps_incr=100,
)

In [ ]:
import hoomd
import gsd.hoomd
for writer in sim.operations.writers:
    if isinstance(writer, hoomd.write.GSD):
        writer.flush()

In [ ]:
sim_visualizer.frame = -1
sim_visualizer.view()